## データベースオブジェクトの準備

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS ontobricks;
CREATE SCHEMA IF NOT EXISTS ontobricks.action_demo_02;

In [0]:
%sql
DROP TABLE IF EXISTS ontobricks.action_demo_02.customer;
CREATE OR REPLACE TABLE ontobricks.action_demo_02.customer (
    customer_id   STRING NOT NULL,
    customer_name STRING,
    status        STRING,
    updated_at    TIMESTAMP
)
 TBLPROPERTIES (
  'delta.enableRowTracking' = 'true'
);

-- Delta Log のバージョンを崩さないために予測最適化を OFF に設定
ALTER TABLE ontobricks.action_demo_02.customer
DISABLE PREDICTIVE OPTIMIZATION;

In [0]:
%sql
INSERT INTO ontobricks.action_demo_02.customer
VALUES
    ('C001', 'Alice', 'ACTIVE', current_timestamp()),
    ('C002', 'Bob',   'ACTIVE', current_timestamp());

In [0]:
%sql
SELECT *
FROM ontobricks.action_demo_02.customer;

## プロシージャの定義

In [0]:
%sql
CREATE OR REPLACE PROCEDURE
    ontobricks.action_demo_02.deactivate_customer_proc(
        IN p_customer_id STRING
    )
LANGUAGE SQL
SQL SECURITY INVOKER
MODIFIES SQL DATA
AS
BEGIN
    UPDATE ontobricks.action_demo_02.customer
       SET status     = 'INACTIVE',
           updated_at = current_timestamp()
     WHERE customer_id = p_customer_id;
END;

In [0]:
%sql
CALL ontobricks.action_demo_02.deactivate_customer_proc('C001');

In [0]:
%sql
SELECT *
FROM ontobricks.action_demo_02.customer
WHERE customer_id = 'C001';

In [0]:
%sql
SELECT *
FROM table_changes(
  'ontobricks.action_demo_02.customer',
  2
)
WHERE customer_id = 'C001';

In [0]:
%sql
UPDATE ontobricks.action_demo_02.customer
SET
    status = 'ACTIVE',
    updated_at = current_timestamp()
WHERE customer_id = 'C001';

In [0]:
%sql
SELECT *
FROM ontobricks.action_demo_02.customer
WHERE customer_id = 'C001';

## 関数の定義

In [0]:
%sql
CREATE OR REPLACE FUNCTION ontobricks.action_demo_02.deactivate_customer_action(
        p_customer_id STRING COMMENT '非アクティブ化する顧客ID'
    )
    RETURNS STRING
    LANGUAGE PYTHON
    SECRETS (ontobricks.action_demo_02.databricks_pat)
    ENVIRONMENT (environment_version = '6')
    COMMENT '指定された顧客を非アクティブ化する OntoBricks Action'
    AS $$
import json
import urllib.request
import urllib.error

from databricks.secrets import get

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
WORKSPACE_URL = "https://dbc-402e780d-a8d9.cloud.databricks.com"
WAREHOUSE_ID = "ba78164d68693318"

# ------------------------------------------------------------
# Authentication
# ------------------------------------------------------------
token = get(
    catalog="ontobricks",
    schema="action_demo_02",
    key="databricks_pat"
)

# ------------------------------------------------------------
# Statement Execution API
# ------------------------------------------------------------
url = f"{WORKSPACE_URL}/api/2.0/sql/statements"
payload = {
    "warehouse_id": WAREHOUSE_ID,
    "catalog": "ontobricks",
    "schema": "action_demo_02",
    "statement": """
        CALL ontobricks.action_demo_02.deactivate_customer_proc(
            :customer_id
        )
    """,
    "parameters": [
        {
            "name": "customer_id",
            "value": p_customer_id,
            "type": "STRING"
        }
    ],
    "wait_timeout": "50s",
    "on_wait_timeout": "CANCEL"
}

request = urllib.request.Request(
    url=url,
    data=json.dumps(payload).encode("utf-8"),
    headers={
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    },
    method="POST"
)

try:
    with urllib.request.urlopen(
        request,
        timeout=60
    ) as response:
        response_body = response.read().decode("utf-8")
        result = json.loads(response_body)

except urllib.error.HTTPError as e:
    error_body = e.read().decode("utf-8")
    raise Exception(
        f"Statement Execution API failed. "
        f"HTTP {e.code}: {error_body}"
    )

except Exception as e:
    raise Exception(
        f"Action invocation failed: {str(e)}"
    )


# ------------------------------------------------------------
# Validate SQL execution
# ------------------------------------------------------------
status = result.get("status", {})
state = status.get("state", "UNKNOWN")

statement_id = result.get(
    "statement_id",
    "UNKNOWN"
)

if state != "SUCCEEDED":
    error = status.get("error", {})
    raise Exception(
        f"Action SQL failed. "
        f"state={state}, "
        f"statement_id={statement_id}, "
        f"error={json.dumps(error)}"
    )


# ------------------------------------------------------------
# Result for OntoBricks
# ------------------------------------------------------------
return json.dumps(
    {
        "success": True,
        "customer_id": p_customer_id,
        "action": "deactivate_customer",
        "statement_id": statement_id,
        "state": state
    }
)
$$;

In [0]:
%sql
SELECT
    ontobricks.action_demo_02.deactivate_customer_action(
        'C001'
    ) AS result;

In [0]:
%sql
SELECT *
FROM ontobricks.action_demo_02.customer
WHERE customer_id = 'C001';

### Databricks Apps に対する権限付与

In [0]:
%sql
-- OntoBricks App の Service Principal Application ID
DECLARE OR REPLACE VARIABLE ontobricks_app_principal STRING
DEFAULT 'cb683fcf-1151-4d99-92f8-5f6982d6093c';

EXECUTE IMMEDIATE
  'GRANT USE CATALOG ON CATALOG ontobricks TO `' 
  || ontobricks_app_principal 
  || '`';

EXECUTE IMMEDIATE
  'GRANT USE SCHEMA ON SCHEMA ontobricks.action_demo_02 TO `' 
  || ontobricks_app_principal 
  || '`';

EXECUTE IMMEDIATE
  'GRANT SELECT ON SCHEMA ontobricks.action_demo_02 TO `' 
  || ontobricks_app_principal 
  || '`';

EXECUTE IMMEDIATE
  'GRANT EXECUTE ON SCHEMA ontobricks.action_demo_02 TO `'
  || ontobricks_app_principal
  || '`';

## クエリ

In [0]:
%sql
SELECT
    customer_id,
    customer_name,
    status,
    updated_at
FROM ontobricks.action_demo_02.customer